# CHSL Coach - fine-tune a speed-solving tutor and export GGUF

Runs on **free Google Colab (T4 GPU)** or Kaggle (GPU T4 x1). Total time is about 40-70 minutes.

1. **Runtime -> Change runtime type -> T4 GPU**
2. Run each cell from top to bottom.
3. At the end you get `chsl-coach-Q4_K_M.gguf` (~1 GB) to copy to your phone and open in the GGUF Lab app.

Before you start, zip the `chsl_tutor` folder from the project as **chsl_tutor.zip**. Cell 2 asks you to upload it.

## 1. Install Unsloth

In [ ]:
%%capture
!pip install unsloth

## 2. Upload `chsl_tutor.zip` and build the dataset
The builder makes ~6000 verified questions. Add your own PYQs with `--extra data/my_questions.jsonl` (format: `data/my_questions_template.jsonl`).

In [ ]:
import os, zipfile
if os.path.basename(os.getcwd()) != "chsl_tutor":
    if not os.path.exists("chsl_tutor"):
        try:
            from google.colab import files
            up = files.upload()                  # choose chsl_tutor.zip
            name = next(iter(up))
        except ImportError:                      # Kaggle: add the zip as a dataset and set its path here
            name = "/kaggle/input/chsl-tutor/chsl_tutor.zip"
        zipfile.ZipFile(name).extractall(".")
    os.chdir("chsl_tutor")
!python build_dataset.py --n 6000
# !python build_dataset.py --n 6000 --extra data/my_questions.jsonl   # <- use this line once you add your own questions

## 3. Settings
`Qwen2.5-1.5B-Instruct` is the right size for a phone (Q4 GGUF ~1 GB, fast replies).
Pick the 3B model if you will run it on a PC or a phone with 8 GB+ RAM.

In [ ]:
BASE_MODEL  = "unsloth/Qwen2.5-1.5B-Instruct"   # or "unsloth/Qwen2.5-3B-Instruct"
MAX_LEN     = 1024
EPOCHS      = 2
LR          = 2e-4
LORA_RANK   = 32
OUT_NAME    = "chsl-coach"
QUANT       = "q4_k_m"                          # q4_k_m = best size/quality for phones; q8_0 = higher quality, 2x size

## 4. Load the model in 4-bit and add LoRA adapters

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_LEN, load_in_4bit=True, dtype=None)

model = FastLanguageModel.get_peft_model(
    model, r=LORA_RANK, lora_alpha=LORA_RANK, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

## 5. Load the dataset

In [ ]:
from datasets import load_dataset
ds = load_dataset("json", data_files={"train": "data/train.jsonl", "validation": "data/val.jsonl"})
def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in batch["messages"]]}
ds = ds.map(to_text, batched=True, remove_columns=ds["train"].column_names)
print(ds)
print(ds["train"][0]["text"][:1200])

## 6. Train (loss is computed only on the tutor's replies)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_LEN,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        warmup_steps=20,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        optim="adamw_8bit",
        logging_steps=20,
        eval_strategy="steps",
        eval_steps=150,
        save_strategy="no",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
stats = trainer.train()
print(stats)

## 7. Try it

In [ ]:
FastLanguageModel.for_inference(model)
SYSTEM = open("system_prompt.txt", encoding="utf-8").read().strip()

def ask(q, max_new_tokens=256):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, temperature=0.2, do_sample=True, top_p=0.9)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

for q in [
    "The price of sugar rises by 20%. By how much percent must a family cut consumption to keep expenditure the same?",
    "A can do a work in 10 days and B in 15 days. In how many days will they finish it together?",
    "Find the angle between the hands of a clock at 4:20.",
    "Spot the error: Each of the students have submitted the form.",
    "Who is the current RBI governor?",
]:
    print("Q:", q); print(ask(q)); print("-" * 60)

## 8. Score on held-out questions
These 300 questions were never used in training. A reply counts as correct if its `Answer:` line has the right option letter or value.

In [ ]:
import json, re
evals = [json.loads(l) for l in open("data/eval.jsonl", encoding="utf-8")]

def norm(s): return re.sub(r"\s+", " ", s.strip().lower())

def is_correct(reply, gold):
    m = re.search(r"Answer:\s*(.*)", reply)
    if not m: return False
    line = norm(m.group(1))
    if gold["letter"]:
        return line.startswith(f"({gold['letter']})")
    return norm(gold["answer"]) in line

N = 150   # raise to 300 for the full set
hits, by_fam = 0, {}
for g in evals[:N]:
    ok = is_correct(ask(g["question"], 200), g)
    hits += ok
    f = by_fam.setdefault(g["family"], [0, 0]); f[0] += ok; f[1] += 1
print(f"Accuracy: {hits}/{N} = {hits/N:.1%}")
for fam, (a, b) in sorted(by_fam.items(), key=lambda x: x[1][0]/x[1][1]):
    print(f"{fam:34s} {a}/{b}")

## 9. Export GGUF
Takes 5-15 minutes (it builds llama.cpp). The output folder has the `.gguf` file.

In [ ]:
model.save_pretrained_gguf(OUT_NAME, tokenizer, quantization_method=QUANT)
import glob; print(glob.glob(f"{OUT_NAME}*/*.gguf") + glob.glob(f"{OUT_NAME}*.gguf") + glob.glob("*.gguf"))

## 10. Save to Google Drive (recommended - the file is ~1 GB)

In [ ]:
import glob, shutil, os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/chsl_coach", exist_ok=True)
for f in set(glob.glob(f"{OUT_NAME}*/*.gguf") + glob.glob(f"{OUT_NAME}*.gguf") + glob.glob("*.gguf")):
    dst = "/content/drive/MyDrive/chsl_coach/" + os.path.basename(f).replace("unsloth", OUT_NAME)
    shutil.copy(f, dst); print("saved", dst)
# Optional: also keep the LoRA adapter to continue training later
model.save_pretrained(f"/content/drive/MyDrive/chsl_coach/{OUT_NAME}-lora")
tokenizer.save_pretrained(f"/content/drive/MyDrive/chsl_coach/{OUT_NAME}-lora")